# Answer Equivalence
Test whether two answers (e.g., the LLM's and Eedi's) are semantically equivalent under LLM-as-a-judge.
Also evaluate the performance of LLM-as-a-judge.

We do this on the student simulation traces, so you need to run the error-simulation notebook first.

In [2]:
import pandas as pd
import os
from openai import OpenAI
from tqdm import tqdm

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

from src.datasets import get_or_create_dataset
from src.equality import SemanticEqualityChecker
from src.model_configurations import gpt_4_1_mini_det_config

### Agreement with Human Annotations

In [ ]:
df = pd.read_csv("eedi_data/sim_results/annotated/naive-deepseek-reasoner.csv")
eedi_dataset = get_or_create_dataset("eedi_data")

In [ ]:
df_fn = df[df["match_annot"] & ~df["match_llm"]]
df_fp = df[~df["match_annot"] & df["match_llm"]]

In [ ]:
for i,r in df_fp.sample(min(3, len(df_fp))).iterrows():
    print("\n\n")
    problem = r["problem"]
    groundtruth = r["groundtruth"]
    llm = r["llm"]

    print(f"<math problem> {problem} </math problem>")
    print(f"<answer_1> {groundtruth} </answer_1>")
    print(f"<answer_2> {llm} </answer_2>")
    print(f"<judgement> different </judgement>")

for i,r in df_fn.sample(min(3, len(df_fn))).iterrows():
    print("\n\n")
    problem = r["problem"]
    groundtruth = r["groundtruth"]
    llm = r["llm"]
    
    print(f"<math problem> {problem} </math problem>")
    print(f"<answer_1> {groundtruth} </answer_1>")
    print(f"<answer_2> {llm} </answer_2>")
    print(f"<judgement> same </judgement>")

In [ ]:
equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))
semantic_equality_checker = SemanticEqualityChecker(equality_client, equality_model_config)

In [ ]:
gpt_4_1_labels = []
for i,row in tqdm(list(df.iterrows())):
    lbl = semantic_equality_checker.is_equal(row["problem"], row["groundtruth"], row["llm"])
    gpt_4_1_labels.append(lbl)

In [ ]:
y_true = df["match_annot"].astype(bool).to_numpy()
y_pred = [bool(x) for x in gpt_4_1_labels]

# Confusion matrix: tn, fp, fn, tp
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Negatives (TN): {tn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")